# PURVA judge committee — Kaggle session recipe

Run **one model per Kaggle session**. Kaggle's free tier gives roughly
30 GPU-hours/week on T4x2 with a 9-hour wall-clock limit per session; a
single 8B-14B judge over the 1,000-sentence pilot should take well under an
hour, but sessions can still be killed by the wall clock, a quota reset, or
an idle timeout — budget conservatively and expect interruptions.

- Set `MODEL_NAME` in the first code cell to one of the
  `purva.committee.models.REGISTRY` keys (`llama-3.1-8b`, `gemma-2-9b`,
  `qwen2.5-14b`, `mistral-nemo-12b`, `aya-expanse-8b`, `indic`) and re-run
  this notebook as a separate session for each judge.
- `run_judge.py` is id-resumable: if a session is killed mid-run,
  re-running the same cells with the same `MODEL_NAME` picks up from
  `data/committee/{model}__{prompt}.jsonl` and skips already-labeled ids —
  it never restarts from zero.
- The pilot set (`data/pilot_set.jsonl`) must be identical across every
  judge. `make_pilot_set.py` refuses to overwrite an existing file and is
  otherwise fully deterministic (seed 42) given the same input corpus, so
  re-running cell 4 in a fresh session reproduces the same 1,000 sentences.
- Attach the `purva-corpus` Kaggle Dataset (containing `corpus_lid.jsonl`)
  to this notebook before running cell 3.
- Add your Hugging Face token as a Kaggle **Secret** named `HF_TOKEN`
  (Add-ons -> Secrets) before running cell 1 — required to download gated
  repos such as Llama and Gemma.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# One judge per session — change this and re-run for each committee member.
# Valid values: llama-3.1-8b, gemma-2-9b, qwen2.5-14b, mistral-nemo-12b,
# aya-expanse-8b, indic (see purva/committee/models.py REGISTRY).
MODEL_NAME = "llama-3.1-8b"

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")


In [ ]:
!git clone https://github.com/shodhx/purva.git
%cd purva
!pip install -q -r requirements-kaggle.txt


In [ ]:
import shutil
from pathlib import Path

Path("data").mkdir(exist_ok=True)
src = Path("/kaggle/input/purva-corpus")

# corpus_lid.jsonl is the GlotLID-labeled corpus (purva/lid2/glotlid_runner.py
# output); pilot_set.jsonl is copied in too if a previous session already
# drew it, so make_pilot_set.py's refuse-overwrite guard reuses it instead
# of drawing a fresh (still-deterministic, but redundant) sample.
for name in ["corpus_lid.jsonl", "pilot_set.jsonl"]:
    candidate = src / name
    if candidate.exists():
        shutil.copy(candidate, f"data/{name}")


In [ ]:
!python -m purva.committee.make_pilot_set
!python -m purva.committee.run_judge --model $MODEL_NAME --pilot


In [ ]:
import shutil
from pathlib import Path

out_dir = Path("/kaggle/working")
shutil.copytree("data/committee", out_dir / "committee", dirs_exist_ok=True)
shutil.copy("data/pilot_set.jsonl", out_dir / "pilot_set.jsonl")

print("copied to /kaggle/working for download:")
for p in sorted((out_dir / "committee").glob("*.jsonl")):
    print(" ", p)
print(" ", out_dir / "pilot_set.jsonl")
